# Adaptive normalization grid — $B^+\to K^-K^+K^+$ with $\phi(1020)$

This notebook stress-tests the dynamics-aware adaptive Dalitz grid with a deliberately difficult case: the very narrow $\phi(1020)\to K^+K^-$ contribution close to the $K^+K^-$ threshold.

The adaptive algorithm does **not** use the concepts of resonance mass or width. It receives the complex dynamical amplitude as an arbitrary function on the Dalitz plot and refines cells where that function is poorly resolved. The same mechanism can therefore be used later for dispersive amplitudes, splines, K-matrix terms, tabulated amplitudes, etc.

Because the two $K^+$ mesons are identical, a $\phi(1020)$ declared in the $(K^-,K^+)$ pair is automatically symmetrized and produces narrow structures in both $s_{12}$ and $s_{13}$.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    AdaptiveDalitzGrid, DalitzGrid, DecayChannel, DecayModel,
    RealImag, Resonance, enable_x64,
)
from dalitzplotfitter.kinematics import dalitz_s13_limits

enable_x64()


## 1. Channel and narrow $\phi(1020)$ model

Particle masses, the $\phi(1020)$ pole mass, width, and spin are taken through the existing `particle`-based model machinery. The meson radii are kept fixed.


In [ ]:
channel = DecayChannel("B+", ("K-", "K+", "K+"))

model = DecayModel(
    channel,
    [
        Resonance(
            "phi(1020)",
            (0, 1),
            RealImag(1.0, 0.0),
            resonance_radius=3.0,
            parent_radius=3.0,
        )
    ],
    normalize_components=False,
)

phi_component = model.amplitude_model.components[0]

def phi_probe(data):
    # Completely generic adaptive-grid interface: a complex function of Dalitz points.
    return phi_component.function(data, None)

print("parent mass    =", channel.parent_mass, "GeV")
print("daughter masses=", channel.daughter_masses, "GeV")
print("component      =", phi_component.name)


## 2. Build a regular grid and the adaptive grid

The regular grid uses the existing equal-area mapping. The adaptive grid starts from a modest base mesh and recursively splits only cells where the $\phi$ dynamics varies rapidly.

`base_resolution` is still important: it is the **discovery scale**. A feature narrower than every initial cell can only trigger refinement if one of the centre/quarter probe points encounters it. For very narrow unknown structures, the base grid therefore cannot be arbitrarily coarse.


In [ ]:
REGULAR_N = 120
regular = DalitzGrid(
    channel.parent_mass, channel.daughter_masses, resolution=REGULAR_N
).sample()

adaptive_builder = AdaptiveDalitzGrid(
    channel.parent_mass,
    channel.daughter_masses,
    base_resolution=48,
    max_depth=5,
    tolerance=0.08,
    max_cells=1_000_000,
)
adaptive = adaptive_builder.build((phi_probe,))
adaptive_sample = adaptive.sample

print(f"regular grid : {REGULAR_N} x {REGULAR_N} = {regular.size:,} points")
print(f"adaptive grid: {adaptive.size:,} leaf cells")
print("maximum adaptive depth =", int(jnp.max(adaptive.depth)))

for depth in range(int(jnp.max(adaptive.depth)) + 1):
    count = int(jnp.sum(adaptive.depth == depth))
    print(f"  depth {depth}: {count:,} cells")


## 3. Full Dalitz plot: where did the algorithm refine?

Points are coloured by refinement depth. The expectation is that the deepest cells concentrate along the narrow $\phi(1020)$ bands instead of being distributed uniformly over phase space.


In [ ]:
M = channel.parent_mass
m1, m2, m3 = channel.daughter_masses
s12_min = (m1 + m2)**2
s12_max = (M - m3)**2
boundary_s12 = jnp.linspace(s12_min, s12_max, 2500)
boundary_low, boundary_high = dalitz_s13_limits(
    boundary_s12, mother_mass=M, masses=channel.daughter_masses
)

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(
    np.asarray(adaptive_sample.s12),
    np.asarray(adaptive_sample.s13),
    c=np.asarray(adaptive.depth),
    s=2.0,
    alpha=0.75,
    rasterized=True,
)
ax.plot(np.asarray(boundary_s12), np.asarray(boundary_low), linewidth=1.2)
ax.plot(np.asarray(boundary_s12), np.asarray(boundary_high), linewidth=1.2)
ax.set_xlabel(r"$s_{12}=m^2(K^-K^+_1)\;[\mathrm{GeV}^2]$")
ax.set_ylabel(r"$s_{13}=m^2(K^-K^+_2)\;[\mathrm{GeV}^2]$")
ax.set_title(r"Adaptive grid for $B^+\to K^-K^+K^+$, probed by $\phi(1020)$")
fig.colorbar(sc, ax=ax, label="refinement depth")
plt.show()


## 4. Zoom near the low-mass corner

The $\phi(1020)$ lies only slightly above the $K^+K^-$ threshold. This is precisely the region where the original equal-area grid has relatively coarse spacing in the invariant-mass direction.


In [ ]:
phi_mass2 = 1.019461**2  # only for choosing the plotting window
window = 0.10

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(
    np.asarray(adaptive_sample.s12),
    np.asarray(adaptive_sample.s13),
    c=np.asarray(adaptive.depth),
    s=5.0,
    alpha=0.8,
    rasterized=True,
)
ax.axvline(phi_mass2, linestyle="--", linewidth=1.0)
ax.axhline(phi_mass2, linestyle="--", linewidth=1.0)
ax.set_xlim(max(s12_min, phi_mass2-window), phi_mass2+window)
ax.set_ylim(max(s12_min, phi_mass2-window), phi_mass2+window)
ax.set_xlabel(r"$s_{12}\;[\mathrm{GeV}^2]$")
ax.set_ylabel(r"$s_{13}\;[\mathrm{GeV}^2]$")
ax.set_title(r"Adaptive refinement around the $\phi(1020)$ bands")
fig.colorbar(sc, ax=ax, label="refinement depth")
plt.show()


## 5. Compare local point density: regular versus adaptive

This comparison uses the actual integration points. The adaptive grid should spend substantially more of its point budget around the narrow $\phi$ bands.


In [ ]:
zoom_lo = max(s12_min, phi_mass2 - 0.06)
zoom_hi = phi_mass2 + 0.06

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)

axes[0].scatter(
    np.asarray(regular.s12), np.asarray(regular.s13),
    s=3.0, alpha=0.65, rasterized=True,
)
axes[0].set_title(f"regular equal-area grid ({regular.size:,} points)")

axes[1].scatter(
    np.asarray(adaptive_sample.s12), np.asarray(adaptive_sample.s13),
    s=3.0, alpha=0.65, rasterized=True,
)
axes[1].set_title(f"adaptive grid ({adaptive.size:,} points)")

for ax in axes:
    ax.axvline(phi_mass2, linestyle="--", linewidth=1.0)
    ax.axhline(phi_mass2, linestyle="--", linewidth=1.0)
    ax.set_xlim(zoom_lo, zoom_hi)
    ax.set_ylim(zoom_lo, zoom_hi)
    ax.set_xlabel(r"$s_{12}\;[\mathrm{GeV}^2]$")
axes[0].set_ylabel(r"$s_{13}\;[\mathrm{GeV}^2]$")
plt.show()


## 6. Weight sanity check

Adaptive cells have unequal physical areas, so their stored weights are no longer identical. Under the package convention `mean(weights * f)`, however, integrating the constant function must still reproduce the total Dalitz area.


In [ ]:
reference_area = float(
    DalitzGrid(channel.parent_mass, channel.daughter_masses, resolution=600).area
)
adaptive_area = float(jnp.mean(adaptive_sample.weights))

print(f"reference Dalitz area = {reference_area:.12f} GeV^4")
print(f"adaptive integral 1    = {adaptive_area:.12f} GeV^4")
print(f"relative difference    = {(adaptive_area/reference_area - 1.0):+.3e}")
print(f"minimum weight         = {float(jnp.min(adaptive_sample.weights)):.6e}")
print(f"maximum weight         = {float(jnp.max(adaptive_sample.weights)):.6e}")


## 7. Integration convergence for the narrow $\phi$ dynamics

We now compare the integral of the **raw dynamical intensity** $|F_\phi|^2$ using regular grids of increasing resolution and the adaptive grid. A fine deterministic grid is used only as a numerical reference for this diagnostic.


In [ ]:
def integral_on(sample):
    values = phi_probe(sample.as_dict())
    return float(jnp.mean(sample.weights * jnp.abs(values)**2))

REFERENCE_N = 700
reference_sample = DalitzGrid(
    channel.parent_mass, channel.daughter_masses, resolution=REFERENCE_N
).sample()
reference_integral = integral_on(reference_sample)

rows = []
for n in [30, 50, 80, 120, 180, 250]:
    sample = DalitzGrid(
        channel.parent_mass, channel.daughter_masses, resolution=n
    ).sample()
    value = integral_on(sample)
    rows.append((f"regular {n}x{n}", sample.size, value, value/reference_integral - 1.0))

adaptive_integral = integral_on(adaptive_sample)
rows.append(("adaptive", adaptive.size, adaptive_integral, adaptive_integral/reference_integral - 1.0))

print(f"reference: {REFERENCE_N}x{REFERENCE_N}, integral = {reference_integral:.12e}\n")
print(f"{'grid':>18s} {'points':>12s} {'integral':>18s} {'relative error':>16s}")
for label, points, value, error in rows:
    print(f"{label:>18s} {points:12,d} {value:18.10e} {error:+16.5e}")


## 8. Error versus number of integration points

A successful adaptive strategy should reach a given integration accuracy with fewer total points than a globally refined grid.


In [ ]:
regular_rows = rows[:-1]
regular_points = np.asarray([row[1] for row in regular_rows])
regular_error = np.abs(np.asarray([row[3] for row in regular_rows]))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(regular_points, regular_error, marker="o", label="regular equal-area grids")
ax.scatter([adaptive.size], [abs(rows[-1][3])], marker="*", s=160, label="adaptive grid")
ax.set_xlabel("number of integration points")
ax.set_ylabel("absolute relative error")
ax.set_title(r"Integration efficiency for narrow $\phi(1020)$ dynamics")
ax.legend()
plt.show()


## 9. Inspect the cells in auxiliary coordinates

The refinement is actually performed in the equal-area $(u,v)$ square. This is convenient because the mapping to the physical Dalitz plot has constant Jacobian. Deep refinement should form narrow regions corresponding to the $\phi$ bands after transformation.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(
    np.asarray(adaptive.u), np.asarray(adaptive.v),
    c=np.asarray(adaptive.depth), s=2.5, alpha=0.75, rasterized=True,
)
ax.set_xlabel(r"$u$")
ax.set_ylabel(r"$v$")
ax.set_title("Adaptive leaf-cell centres in equal-area coordinates")
fig.colorbar(sc, ax=ax, label="refinement depth")
plt.show()


## Conclusions to check

When this notebook is run, the important questions are:

1. Does the refinement lock onto both symmetrized $\phi(1020)$ bands?
2. Is the low-$K^+K^-$-mass region significantly denser than in a regular grid with comparable total point count?
3. Does `mean(weights)` reproduce the physical Dalitz area after recursive splitting?
4. For the integral of $|F_\phi|^2$, does the adaptive grid approach the fine-grid reference with fewer points?
5. Is `base_resolution=48` sufficient to *discover* the narrow $\phi$, or should the base discovery grid be denser?

The last point is important: adaptivity can resolve a feature extremely well **after it is detected**, but no purely local refinement algorithm can guarantee discovery of an arbitrarily narrow feature that falls between all of its initial probe points. This notebook is intended to quantify that limitation for a realistic amplitude-analysis example.
